<a href="https://colab.research.google.com/github/tharun-ragu22/rideshare-predictor/blob/main/RideshareML_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load user input

In [ ]:
month = 6
day = 17
hour = 19

pickup_ward = 10
dropoff_ward = 10

## Load model

In [ ]:
import pickle
import lightgbm as lgb

with open('lgbm_model.pkl', 'rb') as f:
  model = pickle.load(f)

print(model.get_params())

{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.8641881413691186, 'importance_type': 'split', 'learning_rate': 0.13402767422779557, 'max_depth': 8, 'min_child_samples': 100, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 1000, 'n_jobs': None, 'num_leaves': 167, 'objective': None, 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0019963873222314504, 'subsample': 0.7398417802025447, 'subsample_for_bin': 200000, 'subsample_freq': 1, 'device': 'gpu', 'verbosity': -1}


In [ ]:
import json
with open('dump.json', 'w') as f:
  f.write(json.dumps(model.booster_.dump_model()))



## Get time info


If day of year is after today, take year as last year

In [ ]:
from datetime import datetime, timedelta
def get_query_year(month: int, day: int, hour: int, today : datetime = datetime.now()):

  year = today.year - (1 if today.date() <= datetime(today.year, month, day).date() else 0)
  return year

print(get_query_year(2,2,12))

print(get_query_year(12,2,12))
today = datetime.now()

print(get_query_year(today.month, today.day, today.hour))




query_year = get_query_year(month, day, hour)
print(get_query_year(month, day, hour))

query_datetime = datetime(query_year, month, day, hour)

day_of_week = query_datetime.weekday()
print(day_of_week)

2026
2025
2025
2025
1


## Weather info

In [ ]:
!pip install env_canada

In [ ]:
import pandas as pd
import asyncio
import nest_asyncio
from env_canada import ECHistoricalRange
from env_canada.ec_historical import get_historical_stations

from typing import Tuple

# Patch the event loop to allow nested loops
nest_asyncio.apply()




async def fetch_weather_data(query_datetime: datetime)-> Tuple[int, int]:

    num_tries = 0
    while True:
      coordinates = ["43.643233", "-79.385993"]

      stations_data = await get_historical_stations(
          coordinates, start_year=2018, end_year=2026, radius=200, limit=100
      )

      stations = pd.DataFrame(stations_data).T

      ec = ECHistoricalRange(
          station_id=int(stations.iloc[0, 2]),
          timeframe="hourly",
          daterange=(query_datetime, query_datetime + timedelta(hours=1)),
      )

      ec.get_data()
      df = ec.df

      return (
          df.iloc[0]['Precip. Amount (mm)'],
          df.iloc[0]['Temp (°C)'],
          df.iloc[0]['Visibility (km)'],


      )



# Use the loop directly to avoid top-level await SyntaxError
loop = asyncio.get_event_loop()
precipitation, temp, visibility = loop.run_until_complete(fetch_weather_data(query_datetime))
(precipitation, temp, visibility)
# print(weather_df.columns)

(np.float64(0.0), np.float64(19.2), np.float64(16.1))

## Events

In [ ]:
import random

event_in_pickup_ward = 0
event_type = 0
audience = 0

if pickup_ward == 10 and hour >= 19 and hour <= 22:

  import requests
  from google.colab import userdata

  RAPTORS_ID = 28


  BALLDONTLIE_API_KEY = userdata.get('BALLDONTLIE_API_KEY')
  url = 'https://api.balldontlie.io/v1/games'

  headers = {
      "Authorization": f"Bearer {BALLDONTLIE_API_KEY}"
  }

  params = {
      "team_ids[]": RAPTORS_ID,
      "start_date": query_datetime.strftime("%Y-%m-%d"),
      "end_date": query_datetime.strftime("%Y-%m-%d"),


  }

  response = requests.get(url, headers=headers, params=params)

  data = None if not response.ok else response.json()

  if len(data['data']) != 0:
    print(data['data'][0]['home_team']['id'] == RAPTORS_ID)
    print(data['data'][0]['home_team']['id'])
    if data['data'][0]['home_team']['id'] == RAPTORS_ID:
      event_in_pickup_ward = 1
      event_type = 1
      audience = random.randint(18500, 19801)






In [ ]:
if pickup_ward == 10 and hour >= 19 and hour <= 22:
  BLUE_JAYS_ID = 29

  BALLDONTLIE_API_KEY = userdata.get('BALLDONTLIE_API_KEY')
  url = 'https://api.balldontlie.io/mlb/v1/games'

  headers = {
      "Authorization": f"Bearer {BALLDONTLIE_API_KEY}"
  }

  params = {
      "team_ids[]": BLUE_JAYS_ID,
      "dates[]": query_datetime.strftime("%Y-%m-%d"),


  }


  response = requests.get(url, headers=headers, params=params)

  data = None if not response.ok else response.json()
  if data and len(data['data']) != 0:
    print(data['data'][0]['home_team']['id'] == BLUE_JAYS_ID)
    print(data['data'][0]['home_team']['id'])

    if data['data'][0]['home_team']['id'] == BLUE_JAYS_ID:
      event_in_pickup_ward = 1
      event_type = 1
      audience = random.randint(35000, 40000)

True
29


## Make prediction

In [ ]:
data = [{
    "pickup_ward": pickup_ward,
    "dropoff_ward": dropoff_ward,
    "Temp_(°C)": temp,
    "Precip._Amount_(mm)": precipitation,
    "Visibility_(km)": event_in_pickup_ward,
    "event_in_pickup_ward": event_in_pickup_ward,
    "event_type": event_type,
    "audience": audience,
    "hour": hour,
    "day_of_week": day_of_week,
    "day": day,
    "month": month,
    "year": query_year
}]

predict_df = pd.DataFrame(data)

prediction = model.predict(predict_df)

print(prediction)

[12.7934208]


In [ ]:
data

[{'pickup_ward': 10,
  'dropoff_ward': 10,
  'Temp_(°C)': np.float64(19.2),
  'Precip._Amount_(mm)': np.float64(0.0),
  'Visibility_(km)': 1,
  'event_in_pickup_ward': 1,
  'event_type': 1,
  'audience': 38214,
  'hour': 19,
  'day_of_week': 1,
  'day': 17,
  'month': 6,
  'year': 2025}]